In [1]:
#/////////////////////////////////////////////////////////
# preprocessing data.csv
#////////////////////////////////////////////////////////

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from pyspark.sql.types import IntegerType
from pyspark.sql.types import TimestampType 

spark = SparkSession.builder.appName("MusicRecommender").getOrCreate()

data = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/data.csv")

data = data.dropna()

# Liste des colonnes à convertir
columns_to_conver_double = ['valence', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness',
                  'loudness', 'speechiness', 'tempo']
for col_name in columns_to_conver_double:
    data = data.withColumn(col_name, col(col_name).cast(DoubleType()))

columns_to_conver_int = ['year','duration_ms','explicit','key','mode','popularity']
for col_name in columns_to_conver_int:
    data = data.withColumn(col_name, col(col_name).cast(IntegerType()))

columns_to_conver_timestamp = ['release_date']
for col_name in columns_to_conver_timestamp:
    data = data.withColumn(col_name, col(col_name).cast(TimestampType()))

data.printSchema()


root
 |-- valence: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- release_date: timestamp (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)



In [3]:
#/////////////////////////////////////////////////////////////////
# preprocessing data_by_genres.csv
#///////////////////////////////////////////////////////////////////

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

spark = SparkSession.builder.appName("MusicRecommender").getOrCreate()
data_by_genres = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/data_by_genres.csv")

data_by_genres = data_by_genres.dropna()

    

In [5]:
#///////////////////////////////////////////////////////////////
# user inputs
#//////////////////////////////////////////////////////////////

In [6]:
inputs = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://hadoop-namenode-1:8020/data/inputs_1.csv")
inputs.show(10)


+-------+----+------------+--------------------+-------------------+-----------+------------------+--------+--------------------+------------------+---+--------+------------------+----+--------------------+----------+-------------------+-----------+------------------+
|valence|year|acousticness|             artists|       danceability|duration_ms|            energy|explicit|                  id|  instrumentalness|key|liveness|          loudness|mode|                name|popularity|       release_date|speechiness|             tempo|
+-------+----+------------+--------------------+-------------------+-----------+------------------+--------+--------------------+------------------+---+--------+------------------+----+--------------------+----------+-------------------+-----------+------------------+
|  0.898|1994|       0.321|['Jason Weaver', ...| 0.6890000000000001|     170880|0.5329999999999999|       0|0qxtQ8rf3W1nId3D2...|           7.37E-5|  6|  0.0958|           -14.205|   1|"I Just 

In [7]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col

# 1. Définir les colonnes de features
features = ['acousticness', 'danceability', 'duration_ms', 'energy',
            'instrumentalness', 'liveness', 'loudness', 'speechiness',
            'tempo', 'valence', 'popularity', 'key', 'mode']

data = data.dropna(subset=features)
inputs = inputs.dropna(subset=features)

# 2. Vectorisation
assembler = VectorAssembler(inputCols=features, outputCol="features")
data = assembler.transform(data)
inputs = assembler.transform(inputs)

# 3. Standardisation
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scaler_model = scaler.fit(data)
data = scaler_model.transform(data)
inputs = scaler_model.transform(inputs)

# 4. KMeans clustering
kmeans = KMeans(k=10, seed=42, featuresCol="scaledFeatures", predictionCol="cluster")
kmeans_model = kmeans.fit(data)
data = kmeans_model.transform(data)

# 5. Prédiction des clusters pour les musiques de l'utilisateur
inputs = kmeans_model.transform(inputs)

# 6. Extraire les clusters préférés (top 2)
inputs.createOrReplaceTempView("user_tracks")
data.createOrReplaceTempView("all_tracks")

top_clusters = spark.sql("""
SELECT cluster, COUNT(*) AS count
FROM user_tracks
GROUP BY cluster
ORDER BY count DESC
LIMIT 2
""")
top_clusters.createOrReplaceTempView("top_clusters")

# 7. Recommandation : musiques dans les clusters préférés, non déjà écoutées
recommendations = spark.sql("""
SELECT DISTINCT a.*
FROM all_tracks a
JOIN top_clusters t ON a.cluster = t.cluster
WHERE a.id NOT IN (SELECT id FROM user_tracks)
ORDER BY a.popularity DESC
LIMIT 10
""")

# 8. Affichage des recommandations
recommendations.select("name", "artists", "cluster", "popularity").show(truncate=False)


+------------------------------------+----------------------------------------------------------+-------+----------+
|name                                |artists                                                   |cluster|popularity|
+------------------------------------+----------------------------------------------------------+-------+----------+
|positions                           |['Ariana Grande']                                         |5      |96        |
|WAP (feat. Megan Thee Stallion)     |['Cardi B', 'Megan Thee Stallion']                        |5      |96        |
|What You Know Bout Love             |['Pop Smoke']                                             |5      |96        |
|Holy (feat. Chance The Rapper)      |['Justin Bieber', 'Chance the Rapper']                    |5      |95        |
|Relación - Remix                    |['Sech', 'Daddy Yankee', 'J Balvin', 'ROSALÍA', 'Farruko']|5      |94        |
|Head & Heart (feat. MNEK)           |['Joel Corry', 'MNEK']    